In [ ]:
# Cell 1 — Install dependencies
!pip install openai pandas -q

In [ ]:
# Cell 2 — Imports
from openai import OpenAI
import pandas as pd
import time
import os
import csv
from datetime import datetime

In [ ]:
# ============================================================
# Cell 3 — API client and model definitions
#
# Robustness subjects (3 models, 2 origins):
#   Gemini 3.1 Pro Preview  
#   Qwen3.6 Max Preview
#   Claude Sonnet 4.6
#
# Judge (separate notebook): GPT-5.1
# GPT-5.1 is excluded from subjects to avoid self-evaluation
# bias when it serves as judge in the scoring pipeline.
# ============================================================
from google.colab import userdata
OPENR = userdata.get('OPENR')

client = OpenAI(
    api_key=OPENR,
    base_url="https://openrouter.ai/api/v1"
)

# Verify model strings at https://openrouter.ai/models before running
MODELS = {
    "Gemini-3.1-Pro" : "google/gemini-2.5-flash",
    "Qwen3.6-Max"    : "qwen/qwen3.6-flash",
    "Claude-Sonnet-4.6" : "anthropic/claude-Sonnet-4-6"
}

MODEL_ORIGIN = {
    "Gemini-3.1-Pro" : "US",
    "Qwen3.6-Max"    : "China",
    "Claude-Sonnet-4.6" : "US"
}

print("Models registered:")
for name, mid in MODELS.items():
    print(f"  {name:<22} {mid}  [origin={MODEL_ORIGIN[name]}]")

In [ ]:
# ============================================================
# Cell 4 — Upload raw response CSV and reconstruct PROMPTS
#
# USAGE:
#   1. Run this cell — a file picker dialog will appear
#   2. Select the community raw response CSV from your machine
#   3. PROMPTS dict and COMMUNITY_NAME are set automatically
#
# Expected CSV columns (standard pipeline schema):
#   prompt_id | language | prompt | response | ...
#
# COMMUNITY_NAME is inferred from the filename prefix
# (everything before the first underscore), e.g.:
#   'DaiThai_raw_20260128.csv'  -> COMMUNITY_NAME = 'DaiThai'
#   'Lahu_raw_20260201.csv'     -> COMMUNITY_NAME = 'Lahu'
# ============================================================

from google.colab import files

print("Please upload the community raw response CSV...")
uploaded = files.upload()  # opens file picker dialog

# Retrieve the uploaded filename (expects exactly one file)
assert len(uploaded) == 1, "Please upload exactly one CSV file."
RAW_CSV_NAME = list(uploaded.keys())[0]
print(f"Uploaded: {RAW_CSV_NAME}")

# Read into dataframe
import io
df_source = pd.read_csv(io.BytesIO(uploaded[RAW_CSV_NAME]))

# Verify expected columns are present
required_cols = {"prompt_id", "language", "prompt"}
assert required_cols.issubset(df_source.columns), (
    f"Missing columns: {required_cols - set(df_source.columns)}"
)

# Rebuild PROMPTS dict in the standard {pid: {cn: ..., en: ...}} format
PROMPTS = {}
for _, row in (
    df_source[["prompt_id", "language", "prompt"]]
    .drop_duplicates()
    .iterrows()
):
    pid  = row["prompt_id"]
    lang = "cn" if row["language"] == "Chinese" else "en"
    if pid not in PROMPTS:
        PROMPTS[pid] = {}
    PROMPTS[pid][lang] = row["prompt"]

# Sort A1 -> D3
PROMPTS = dict(sorted(PROMPTS.items()))

# Infer community name from filename prefix for output file naming
COMMUNITY_NAME = RAW_CSV_NAME.split("_")[0]

n_models    = len(MODELS)
n_prompts   = len(PROMPTS)
n_languages = 2
total_q     = n_prompts * n_models * n_languages

print()
print(f"Community      : {COMMUNITY_NAME}")
print(f"Prompts loaded : {n_prompts}")
print(f"Total queries  : {n_prompts} x {n_models} models x {n_languages} languages = {total_q}")
print()
print("Prompt preview:")
for pid, langs in PROMPTS.items():
    cn_preview = langs.get('cn', 'MISSING')[:40]
    en_preview = langs.get('en', 'MISSING')[:40]
    print(f"  {pid}  CN: {cn_preview}")
    print(f"       EN: {en_preview}")

In [ ]:
# ============================================================
# Cell 5 — OpenRouter API helper
# ============================================================

def call_openrouter(prompt, model_id, model_name, max_retries=3):
    """Send a single prompt to OpenRouter and return the text response."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=2000,
                extra_headers={
                    "HTTP-Referer": "https://github.com/ooodddee/Trans-border-Representation-Probe",
                    "X-Title"     : "Trans-border AI Probe - Robustness Check"
                }
            )
            return response.choices[0].message.content
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"  Retry {attempt + 1}/{max_retries} [{model_name}]: {e}")
                time.sleep(5)
            else:
                return f"ERROR: {e}"

# Smoke test — verify all three models respond
print("Testing API connections...")
for model_name, model_id in MODELS.items():
    t = call_openrouter("Hello, respond with one word.", model_id, model_name)
    status = "✅" if t and not str(t).startswith("ERROR") else "❌"
    print(f"  {status} {model_name:<20}: {str(t)[:60]}")

In [ ]:
# ============================================================
# Cell 6 — Data collection
#
# Loop order: prompt → model → language
# Matches primary notebook loop order for consistency
# Total: n_prompts × 3 models × 2 languages
# ============================================================

results = []
total   = len(PROMPTS) * len(MODELS) * 2
current = 0

print("=" * 65)
print(f"Trans-border Representation Probe — {COMMUNITY_NAME} (Robustness)")
print(f"Subjects : {list(MODELS.keys())}")
print(f"Queries  : {total}")
print("=" * 65)

for prompt_id, prompt_data in PROMPTS.items():
    for model_name, model_id in MODELS.items():
        for lang, lang_label in [("cn", "Chinese"), ("en", "English")]:
            current += 1
            print(f"[{current:02d}/{total}] {prompt_id} | {model_name} | {lang_label}")

            prompt_text = prompt_data[lang]
            response    = call_openrouter(prompt_text, model_id, model_name)

            results.append({
                "prompt_id"   : prompt_id,
                "category"    : prompt_id[0],          # A / B / C / D
                "model"       : model_name,
                "model_origin": MODEL_ORIGIN[model_name],
                "model_tier"  : "robustness",
                "language"    : lang_label,
                "prompt"      : prompt_text,
                "response"    : response,
                "timestamp"   : datetime.now().isoformat()
            })

            time.sleep(1)  # rate limiting

df = pd.DataFrame(results)
print(f"\nCollection complete — {len(df)} responses collected.")

In [ ]:
# ============================================================
# Cell 7 — Save raw responses and download
# Filename: {COMMUNITY_NAME}_robustness_raw_{timestamp}.csv
# ============================================================

filename = f"{COMMUNITY_NAME}_robustness_raw_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df.to_csv(filename, index=False, encoding="utf-8-sig")
print(f"✅ Saved: {filename}")
print(f"   Rows   : {len(df)}")
print(f"   Columns: {list(df.columns)}")

from google.colab import files
files.download(filename)

In [ ]:
# ============================================================
# Cell 8 — Quick sanity check
# Response count per model × language condition
# All cells should equal n_prompts (e.g. 11)
# Any ERROR entries flagged for rerun
# ============================================================

print("Response count per condition:")
pivot = df.pivot_table(
    index=["model", "model_origin"],
    columns="language",
    values="response",
    aggfunc="count"
)
print(pivot)

# Flag errors
errors = df[df["response"].str.startswith("ERROR", na=False)]
if len(errors) == 0:
    print("\n✅ No errors detected.")
else:
    print(f"\n❌ {len(errors)} error(s) detected:")
    print(errors[["prompt_id", "model", "language", "response"]].to_string())